In [ ]:
# Import packages
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt 

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression # Baseline model
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score  # Metrics loaded
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

In [ ]:
# =========================
# CONFIG
# =========================
DATA_PATH = "data/employee_data.csv"   # change if needed
RANDOM_STATE = 42 # set seed
TARGET_COL = "Current Employee Rating"

# Columns to drop because they are IDs, personally identifying,
# or likely to create leakage / be unhelpful for baseline
DROP_COLS = [
    "EmpID",
    "FirstName",
    "LastName",
    "StartDate",
    "ExitDate",
    "Title",
    "Supervisor",
    "ADEmail",
    "BusinessUnit",
    "EmployeeStatus",
    "TerminationType",
    "TerminationDescription",
    "DOB",
    "DateofHire",
    "Performance Score",   # leakage / close to target / Gonna be too correlated
]

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


# =========================
# HELPER FUNCTIONS
# =========================
def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("/", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("(", "", regex=False)
        .str.replace(")", "", regex=False)
    )
    return df


def parse_dates(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # handle likely date columns if they exist
    possible_date_cols = ["DOB", "DateofHire", "StartDate", "ExitDate"]
    for col in possible_date_cols:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors="coerce")

    return df


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # age
    if "DOB" in df.columns:
        reference_date = pd.Timestamp("2025-01-01")
        df["Age_Years"] = ((reference_date - df["DOB"]).dt.days / 365.25).round(2)

    # tenure
    if "DateofHire" in df.columns:
        reference_date = pd.Timestamp("2025-01-01")
        df["Tenure_Days"] = (reference_date - df["DateofHire"]).dt.days

    # active flag
    if "EmployeeStatus" in df.columns:
        df["Is_Active"] = (df["EmployeeStatus"].astype(str).str.lower() == "active").astype(int)

    return df


def safe_drop_columns(df: pd.DataFrame, cols_to_drop: list) -> pd.DataFrame:
    existing_cols = [c for c in cols_to_drop if c in df.columns]
    return df.drop(columns=existing_cols)


def build_preprocessor(X: pd.DataFrame, polynomial_numeric: bool = False) -> ColumnTransformer:
    numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

    numeric_steps = [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
    if polynomial_numeric:
        numeric_steps.append(("poly", PolynomialFeatures(degree=2, include_bias=False)))

    numeric_pipeline = Pipeline(steps=numeric_steps)

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        sparse_threshold=0,
    )

    return preprocessor


def evaluate_classification(y_true, y_pred) -> dict:
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="weighted")
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")

    return {
        "accuracy": float(acc),
        "f1_weighted": float(f1),
        "precision_weighted": float(precision),
        "recall_weighted": float(recall),
    }


def build_validation_diagnostics(y_true, model_predictions: dict) -> dict:
    labels = sorted(y_true.dropna().unique().tolist())
    diagnostics = {
        "true_class_distribution": y_true.value_counts().sort_index().astype(int).to_dict(),
        "models": {},
    }

    for model_name, y_pred in model_predictions.items():
        pred_series = pd.Series(y_pred)
        diagnostics["models"][model_name] = {
            "predicted_class_distribution": pred_series.value_counts().sort_index().astype(int).to_dict(),
            "confusion_matrix_labels": labels,
            "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
            "classification_report": classification_report(
                y_true,
                y_pred,
                labels=labels,
                output_dict=True,
                zero_division=0,
            ),
        }

    return diagnostics

In [ ]:
# =========================
# MAIN
# =========================
def main():
    start_time = time.time()

    # 1. Load data
    df = pd.read_csv(DATA_PATH)

    # 2. Parse and feature engineer (Cleaning data + mutate data)
    df = parse_dates(df)
    df = engineer_features(df)

    # 3. Drop rows with missing target
    if TARGET_COL not in df.columns:
        raise ValueError(f"Target column '{TARGET_COL}' not found in dataset.")

    df = df[df[TARGET_COL].notna()].copy() # Check validation

    # 4. Define X and y
    y = df[TARGET_COL]
    X = safe_drop_columns(df, DROP_COLS + [TARGET_COL])

    # Also remove any datetime columns after feature engineering because we dont need them anymore
    datetime_cols = X.select_dtypes(include=["datetime64[ns]", "datetime64"]).columns.tolist()
    if datetime_cols:
        X = X.drop(columns=datetime_cols)

    # (Special) Check the distribution of target variable => Normal dist?
    print(df[TARGET_COL].value_counts(normalize=True))
    
    plt.hist(df[TARGET_COL], bins=[0.5,1.5,2.5,3.5,4.5,5.5])
    plt.xticks([1,2,3,4,5])
    plt.show()
    
    # 5. Deterministic split
    # 70 train / 15 val / 15 test
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X, y, test_size=0.15, random_state=RANDOM_STATE
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=0.1764705882,  # gives ~15% of full sample
        random_state=RANDOM_STATE
    )

    X_train.to_csv("data/processed/X_train.csv")
    y_train.to_csv("data/processed/y_train.csv")
    X_val.to_csv("data/processed/X_val.csv")
    y_val.to_csv("data/processed/y_val.csv")

    # 6. Build preprocessor + logistic regression baseline
    preprocessor = build_preprocessor(X_train)

    logistic_start_time = time.time()
    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("clf", LogisticRegression(max_iter=1000)),
        ]
    )

    # 7. Train
    model.fit(X_train, y_train)

    # 8. Validate
    val_preds = model.predict(X_val)
    
    val_metrics = evaluate_classification(y_val, val_preds)


    runtime_seconds = time.time() - logistic_start_time # Check timing

    # 9. Print stable evaluator output
    print("=== BASELINE VALIDATION RESULTS ===")
    print(f"Validation ACC: {val_metrics['accuracy']:.4f}")
    print(f"Validation F1:  {val_metrics['f1_weighted']:.4f}")
    print(f"Validation Precision:  {val_metrics['precision_weighted']:.4f}")
    print(f"Validation Recall: {val_metrics['recall_weighted']:.4f}")
    print(f"Runtime (seconds): {runtime_seconds:.4f}")
    print("Test set has been created and held out. Do not use it during search phase.")

    # 10. Tune random forest baseline on training data only
    rf_start_time = time.time()
    rf_pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(X_train)),
            ("clf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        ]
    )

    rf_param_grid = {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [None, 10, 20],
        "clf__min_samples_leaf": [1, 2, 4],
        "clf__max_features": ["sqrt", "log2"],
    }

    rf_search = GridSearchCV(
        estimator=rf_pipeline,
        param_grid=rf_param_grid,
        scoring="f1_weighted",
        cv=3,
        n_jobs=1,
    )
    rf_search.fit(X_train, y_train)

    rf_val_preds = rf_search.predict(X_val)
    rf_val_metrics = evaluate_classification(y_val, rf_val_preds)
    rf_runtime_seconds = time.time() - rf_start_time

    print("=== RANDOM FOREST VALIDATION RESULTS ===")
    print(f"Best params: {rf_search.best_params_}")
    print(f"Best CV F1: {rf_search.best_score_:.4f}")
    print(f"Validation ACC: {rf_val_metrics['accuracy']:.4f}")
    print(f"Validation F1:  {rf_val_metrics['f1_weighted']:.4f}")
    print(f"Validation Precision:  {rf_val_metrics['precision_weighted']:.4f}")
    print(f"Validation Recall: {rf_val_metrics['recall_weighted']:.4f}")
    print(f"Runtime (seconds): {rf_runtime_seconds:.4f}")

    # 11. Tune polynomial logistic regression on training data only
    poly_start_time = time.time()
    poly_pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(X_train, polynomial_numeric=True)),
            ("clf", LogisticRegression(max_iter=2000)),
        ]
    )

    poly_param_grid = {
        "clf__C": [0.1, 1.0, 10.0],
        "clf__class_weight": [None, "balanced"],
    }

    poly_search = GridSearchCV(
        estimator=poly_pipeline,
        param_grid=poly_param_grid,
        scoring="f1_weighted",
        cv=3,
        n_jobs=1,
    )
    poly_search.fit(X_train, y_train)

    poly_val_preds = poly_search.predict(X_val)
    poly_val_metrics = evaluate_classification(y_val, poly_val_preds)
    poly_runtime_seconds = time.time() - poly_start_time

    print("=== POLYNOMIAL LOGISTIC VALIDATION RESULTS ===")
    print(f"Best params: {poly_search.best_params_}")
    print(f"Best CV F1: {poly_search.best_score_:.4f}")
    print(f"Validation ACC: {poly_val_metrics['accuracy']:.4f}")
    print(f"Validation F1:  {poly_val_metrics['f1_weighted']:.4f}")
    print(f"Validation Precision:  {poly_val_metrics['precision_weighted']:.4f}")
    print(f"Validation Recall: {poly_val_metrics['recall_weighted']:.4f}")
    print(f"Runtime (seconds): {poly_runtime_seconds:.4f}")

    # 12. Tune boosted tree baseline on training data only
    boosted_start_time = time.time()
    boosted_pipeline = Pipeline(
        steps=[
            ("preprocessor", build_preprocessor(X_train)),
            ("clf", GradientBoostingClassifier(random_state=RANDOM_STATE)),
        ]
    )

    boosted_param_grid = {
        "clf__n_estimators": [50, 100],
        "clf__learning_rate": [0.05, 0.1],
        "clf__max_depth": [2, 3],
    }

    boosted_search = GridSearchCV(
        estimator=boosted_pipeline,
        param_grid=boosted_param_grid,
        scoring="f1_weighted",
        cv=3,
        n_jobs=1,
    )
    boosted_search.fit(X_train, y_train)

    boosted_val_preds = boosted_search.predict(X_val)
    boosted_val_metrics = evaluate_classification(y_val, boosted_val_preds)
    boosted_runtime_seconds = time.time() - boosted_start_time

    print("=== BOOSTED TREE VALIDATION RESULTS ===")
    print(f"Best params: {boosted_search.best_params_}")
    print(f"Best CV F1: {boosted_search.best_score_:.4f}")
    print(f"Validation ACC: {boosted_val_metrics['accuracy']:.4f}")
    print(f"Validation F1:  {boosted_val_metrics['f1_weighted']:.4f}")
    print(f"Validation Precision:  {boosted_val_metrics['precision_weighted']:.4f}")
    print(f"Validation Recall: {boosted_val_metrics['recall_weighted']:.4f}")
    print(f"Runtime (seconds): {boosted_runtime_seconds:.4f}")

    # 13. Build validation diagnostics for class-level inspection
    validation_diagnostics = build_validation_diagnostics(
        y_val,
        {
            "LogisticRegression": val_preds,
            "RandomForestClassifier": rf_val_preds,
            "PolynomialLogisticRegression": poly_val_preds,
            "GradientBoostingClassifier": boosted_val_preds,
        },
    )

    print("=== VALIDATION PREDICTED CLASS DISTRIBUTIONS ===")
    for model_name, model_diagnostics in validation_diagnostics["models"].items():
        print(f"{model_name}: {model_diagnostics['predicted_class_distribution']}")

    # 14. Save split sizes for reproducibility
    split_info = {
        "random_state": RANDOM_STATE,
        "target_column": TARGET_COL,
        "n_total": int(len(df)),
        "n_train": int(len(X_train)),
        "n_val": int(len(X_val)),
        "n_test": int(len(X_test)),
        "test_set_policy": "Locked and not used during search phase.",
    }

    with open(RESULTS_DIR / "split_info.json", "w") as f:
        json.dump(split_info, f, indent=2)

    # 14. Save baseline metrics
    output_metrics = {
        "model": "LogisticRegression",
        "validation_metrics": val_metrics,
        "runtime_seconds": runtime_seconds,
    }

    with open(RESULTS_DIR / "baseline_metrics.json", "w") as f:
        json.dump(output_metrics, f, indent=2)

    rf_output_metrics = {
        "model": "RandomForestClassifier",
        "best_params": rf_search.best_params_,
        "best_cv_f1_weighted": float(rf_search.best_score_),
        "validation_metrics": rf_val_metrics,
        "runtime_seconds": rf_runtime_seconds,
    }

    with open(RESULTS_DIR / "random_forest_metrics.json", "w") as f:
        json.dump(rf_output_metrics, f, indent=2)

    poly_output_metrics = {
        "model": "PolynomialLogisticRegression",
        "best_params": poly_search.best_params_,
        "best_cv_f1_weighted": float(poly_search.best_score_),
        "validation_metrics": poly_val_metrics,
        "runtime_seconds": poly_runtime_seconds,
    }

    with open(RESULTS_DIR / "polynomial_logistic_metrics.json", "w") as f:
        json.dump(poly_output_metrics, f, indent=2)

    boosted_output_metrics = {
        "model": "GradientBoostingClassifier",
        "best_params": boosted_search.best_params_,
        "best_cv_f1_weighted": float(boosted_search.best_score_),
        "validation_metrics": boosted_val_metrics,
        "runtime_seconds": boosted_runtime_seconds,
    }

    with open(RESULTS_DIR / "boosted_tree_metrics.json", "w") as f:
        json.dump(boosted_output_metrics, f, indent=2)

    with open(RESULTS_DIR / "validation_diagnostics.json", "w") as f:
        json.dump(validation_diagnostics, f, indent=2)

    # 15. Save experiment log entries
    log_row = pd.DataFrame(
        [
            {
                "experiment_id": 1,
                "model": "LogisticRegression",
                "target": TARGET_COL,
                "validation_metric": "accuracy",
                "validation_accuracy": val_metrics["accuracy"],
                "validation_precision": val_metrics["precision_weighted"],
                "validation_recall": val_metrics["recall_weighted"],
                "runtime_seconds": runtime_seconds,
                "random_state": RANDOM_STATE,
                "notes": "Week 2 reproducible baseline run"
            },
            {
                "experiment_id": 2,
                "model": "RandomForestClassifier",
                "target": TARGET_COL,
                "validation_metric": "f1_weighted",
                "validation_accuracy": rf_val_metrics["accuracy"],
                "validation_precision": rf_val_metrics["precision_weighted"],
                "validation_recall": rf_val_metrics["recall_weighted"],
                "runtime_seconds": rf_runtime_seconds,
                "random_state": RANDOM_STATE,
                "notes": f"GridSearchCV best params: {rf_search.best_params_}"
            },
            {
                "experiment_id": 3,
                "model": "PolynomialLogisticRegression",
                "target": TARGET_COL,
                "validation_metric": "f1_weighted",
                "validation_accuracy": poly_val_metrics["accuracy"],
                "validation_precision": poly_val_metrics["precision_weighted"],
                "validation_recall": poly_val_metrics["recall_weighted"],
                "runtime_seconds": poly_runtime_seconds,
                "random_state": RANDOM_STATE,
                "notes": f"Numeric degree-2 polynomial features; GridSearchCV best params: {poly_search.best_params_}"
            },
            {
                "experiment_id": 4,
                "model": "GradientBoostingClassifier",
                "target": TARGET_COL,
                "validation_metric": "f1_weighted",
                "validation_accuracy": boosted_val_metrics["accuracy"],
                "validation_precision": boosted_val_metrics["precision_weighted"],
                "validation_recall": boosted_val_metrics["recall_weighted"],
                "runtime_seconds": boosted_runtime_seconds,
                "random_state": RANDOM_STATE,
                "notes": f"GridSearchCV best params: {boosted_search.best_params_}"
            }
        ]
    )

    log_path = RESULTS_DIR / "experiment_log.csv"
    if log_path.exists():
        old_log = pd.read_csv(log_path)
        if "experiment_id" in old_log.columns:
            old_log = old_log[~old_log["experiment_id"].isin(log_row["experiment_id"])]
        full_log = pd.concat([old_log, log_row], ignore_index=True)
    else:
        full_log = log_row

    full_log.to_csv(log_path, index=False)

    # 16. Save locked test set indices
    test_indices = pd.DataFrame({"test_index": X_test.index})
    test_indices.to_csv(RESULTS_DIR / "locked_test_indices.csv", index=False)


if __name__ == "__main__":
    main()
